# 🇬🇧 ➜ 🇫🇷 English-to-French Seq2Seq Translation — Colab GPU Training

**Prerequisites:**
1. Set **Runtime → Change runtime type → GPU (T4)**.
2. Upload `fra.txt` (Tatoeba supplement, ~34 MB) to your Google Drive at:
   ```
   My Drive/week_8_project/data/fra.txt
   ```
3. *(Optional)* Upload `en-fr.csv` (Kaggle 8 GB) to the same folder for a larger Kaggle mix.

The notebook will:
- Mount Google Drive
- Load and preprocess the data
- Train with **CUDA GPU acceleration**
- Save `encoder.pt`, `decoder.pt`, `vocab.pkl`, **and a full `translation_model.pkl`** back to Drive

## 1 — Mount Google Drive & verify GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        '⚠️  No GPU detected!\n'
        'Go to Runtime → Change runtime type → Hardware accelerator → GPU'
    )

device = torch.device('cuda')
print('Using device:', device)

## 2 — Configuration

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────
DRIVE_DATA_DIR   = '/content/drive/MyDrive/week_8_project/data'
DRIVE_MODEL_DIR  = '/content/drive/MyDrive/week_8_project/model'

# The file you need to upload to Google Drive:
TATOEBA_TXT = f'{DRIVE_DATA_DIR}/fra.txt'
# (Optional) large Kaggle CSV — leave None if not uploaded
KAGGLE_CSV  = f'{DRIVE_DATA_DIR}/en-fr.csv'   # set to None to skip

# ── Hyper-parameters ──────────────────────────────────────────────
SAMPLE_LIMIT   = 25_000     # total training pairs (increase for better quality)
EPOCHS         = 15
HIDDEN_SIZE    = 256
LEARNING_RATE  = 0.001
PRINT_EVERY    = 2000
SEED           = 42

## 3 — Model definitions (Encoder + Attention + Decoder)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

SOS_token = 0
EOS_token = 1
MAX_LENGTH = 15


class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_p=0.1):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(input_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, input):
        embedded = self.dropout(self.embedding(input))
        output, hidden = self.gru(embedded)
        return output, hidden


class BahdanauAttention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.Wa = nn.Linear(hidden_size, hidden_size)
        self.Ua = nn.Linear(hidden_size, hidden_size)
        self.Va = nn.Linear(hidden_size, 1)

    def forward(self, query, keys):
        scores = self.Va(torch.tanh(self.Wa(query) + self.Ua(keys)))
        scores = scores.squeeze(2).unsqueeze(1)
        weights = F.softmax(scores, dim=-1)
        context = torch.bmm(weights, keys)
        return context, weights


class AttnDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, dropout_p=0.1):
        super().__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.attention = BahdanauAttention(hidden_size)
        self.gru = nn.GRU(2 * hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)
        self.dropout = nn.Dropout(dropout_p)

    def forward_step(self, input, hidden, encoder_outputs):
        embedded = self.dropout(self.embedding(input))
        query = hidden.permute(1, 0, 2)
        context, attn_weights = self.attention(query, encoder_outputs)
        input_gru = torch.cat((embedded, context), dim=2)
        output, hidden = self.gru(input_gru, hidden)
        output = self.out(output)
        return output, hidden, attn_weights

    def forward(self, encoder_outputs, encoder_hidden,
                target_tensor=None, max_len=MAX_LENGTH, device='cpu'):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(
            batch_size, 1, dtype=torch.long, device=device
        ).fill_(SOS_token)
        decoder_hidden = encoder_hidden
        outputs, attentions = [], []

        decode_steps = (
            target_tensor.size(1) if target_tensor is not None else max_len
        )

        for i in range(decode_steps):
            decoder_output, decoder_hidden, attn_weights = self.forward_step(
                decoder_input, decoder_hidden, encoder_outputs
            )
            outputs.append(decoder_output)
            attentions.append(attn_weights)
            if target_tensor is not None:
                decoder_input = target_tensor[:, i].unsqueeze(1)
            else:
                _, topi = decoder_output.topk(1)
                decoder_input = topi.squeeze(-1).detach()

        outputs = torch.cat(outputs, dim=1)
        outputs = F.log_softmax(outputs, dim=-1)
        return outputs, torch.cat(attentions, dim=1)

## 4 — Data loading & preprocessing

In [ ]:
import csv
import os
import re
import random
import unicodedata

import pandas as pd


# ── Text normalisation ─────────────────────────────────────────────
def unicode_to_ascii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

CONTRACTIONS = {
    "i'm": "i am", "you're": "you are", "he's": "he is",
    "she's": "she is", "it's": "it is", "we're": "we are",
    "they're": "they are", "i've": "i have", "you've": "you have",
    "we've": "we have", "they've": "they have", "i'll": "i will",
    "you'll": "you will", "he'll": "he will", "she'll": "she will",
    "we'll": "we will", "they'll": "they will", "don't": "do not",
    "doesn't": "does not", "didn't": "did not", "won't": "will not",
    "can't": "can not", "couldn't": "could not",
    "shouldn't": "should not", "wouldn't": "would not",
    "isn't": "is not", "aren't": "are not", "wasn't": "was not",
    "weren't": "were not", "haven't": "have not",
    "hasn't": "has not", "hadn't": "had not",
}

def normalize_string(s):
    s = unicode_to_ascii(s.lower().strip())
    for contraction, expanded in CONTRACTIONS.items():
        s = re.sub(rf'\b{re.escape(contraction)}\b', expanded, s)
    s = re.sub(r'([.!?])', r' \1', s)
    s = re.sub(r"[^a-zA-Z.!?']+", r' ', s)
    return s.strip()


class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: 'SOS', 1: 'EOS'}
        self.n_words = 2

    def add_sentence(self, sentence):
        for word in sentence.split(' '):
            self.add_word(word)

    def add_word(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1


def filter_pair(p):
    return (
        len(p[0].split(' ')) < MAX_LENGTH
        and len(p[1].split(' ')) < MAX_LENGTH
    )


# ── Essential phrase pairs (hard-coded so the model learns them) ──
ESSENTIAL_PAIRS = [
    ('i am learning french .', 'j apprends le francais .'),
    ('i am learning french', 'j apprends le francais'),
    ('i am learning english .', "j apprends l anglais ."),
    ('hello .', 'salut .'), ('hello', 'salut'),
    ('thank you .', 'merci .'), ('thank you', 'merci'),
    ('how are you ?', 'comment allez vous ?'),
    ('how are you', 'comment allez vous'),
    ('good morning .', 'bonjour .'), ('good night .', 'bonne nuit .'),
    ('i love you .', 'je t aime .'),
    ('where is the bathroom ?', 'ou sont les toilettes ?'),
    ('see you later .', 'a plus tard .'),
    ('what is your name ?', 'quel est votre nom ?'),
    ('my name is tom .', 'mon nom est tom .'),
    ('i am hungry .', "j ai faim ."),
    ('i am tired .', 'je suis fatigue .'),
    ('do you speak english ?', 'parlez vous anglais ?'),
    ('i do not understand .', 'je ne comprends pas .'),
]
ESSENTIAL_REPEAT = 80

In [ ]:
# ── Load Tatoeba fra.txt from Google Drive ─────────────────────────
def load_tatoeba_pairs(path, limit=25000, seed=42):
    all_pairs = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) < 2:
                continue
            pair = [normalize_string(parts[0]), normalize_string(parts[1])]
            if filter_pair(pair):
                all_pairs.append(pair)
    rng = random.Random(seed)
    rng.shuffle(all_pairs)
    return all_pairs[:limit]


# ── (Optional) Load Kaggle CSV ─────────────────────────────────────
def detect_columns(fieldnames):
    lower = {name.lower().strip(): name for name in fieldnames}
    eng_keys = ['english words/sentences', 'english', 'en', 'source', 'src']
    fra_keys = ['french words/sentences', 'french', 'fr', 'target', 'tgt']
    eng_col = next((lower[k] for k in eng_keys if k in lower), None)
    fra_col = next((lower[k] for k in fra_keys if k in lower), None)
    if eng_col and fra_col:
        return eng_col, fra_col
    if len(fieldnames) >= 2:
        return fieldnames[0], fieldnames[1]
    raise ValueError(f'Could not detect English/French columns in: {fieldnames}')


def load_kaggle_pairs(path, sample_limit, seed=42):
    preview = pd.read_csv(path, nrows=0)
    eng_col, fra_col = detect_columns(preview.columns.tolist())
    print(f'  Kaggle columns: {eng_col!r} → {fra_col!r}')
    pairs, raw = [], 0
    for chunk in pd.read_csv(path, usecols=[eng_col, fra_col],
                             chunksize=100_000, dtype=str):
        for eng, fra in zip(chunk[eng_col], chunk[fra_col]):
            raw += 1
            if raw > 2_000_000:
                break
            if pd.isna(eng) or pd.isna(fra):
                continue
            pair = [normalize_string(str(eng)), normalize_string(str(fra))]
            if filter_pair(pair):
                pairs.append(pair)
            if len(pairs) >= sample_limit:
                break
        if len(pairs) >= sample_limit or raw > 2_000_000:
            break
    rng = random.Random(seed)
    rng.shuffle(pairs)
    return pairs[:sample_limit]

In [ ]:
# ── Assemble the training dataset ──────────────────────────────────
print('Loading Tatoeba pairs from Google Drive …')
if not os.path.exists(TATOEBA_TXT):
    raise FileNotFoundError(
        f'❌ fra.txt not found at:\n  {TATOEBA_TXT}\n'
        'Upload it to Google Drive → My Drive/week_8_project/data/fra.txt'
    )

tatoeba_limit = min(
    max(SAMPLE_LIMIT - len(ESSENTIAL_PAIRS) * ESSENTIAL_REPEAT,
        SAMPLE_LIMIT * 3 // 4),
    40_000,
)
tatoeba_pairs = load_tatoeba_pairs(TATOEBA_TXT, limit=tatoeba_limit, seed=SEED)
print(f'  Tatoeba: {len(tatoeba_pairs)} pairs')

# Optional Kaggle supplement
kaggle_pairs = []
if KAGGLE_CSV and os.path.exists(KAGGLE_CSV):
    kaggle_limit = max(SAMPLE_LIMIT - len(tatoeba_pairs), SAMPLE_LIMIT // 8)
    kaggle_pairs = load_kaggle_pairs(KAGGLE_CSV, kaggle_limit, seed=SEED)
    print(f'  Kaggle:  {len(kaggle_pairs)} pairs')
else:
    print('  Kaggle CSV not found — training on Tatoeba only (this is fine).')

# Combine
pairs = tatoeba_pairs + kaggle_pairs
rng = random.Random(SEED)
rng.shuffle(pairs)
pairs = pairs[:SAMPLE_LIMIT]

# Add essential phrases
essential_block = [p for _ in range(ESSENTIAL_REPEAT) for p in ESSENTIAL_PAIRS]
pairs = essential_block + pairs
print(f'  Essential phrases: {len(essential_block)}')
print(f'  Total pairs: {len(pairs)}')

# Build vocabularies
input_lang, output_lang = Lang('eng'), Lang('fra')
for pair in pairs:
    input_lang.add_sentence(pair[0])
    output_lang.add_sentence(pair[1])

print(f'Vocab → eng: {input_lang.n_words} words, fra: {output_lang.n_words} words')

## 5 — Train / test split & helper tensors

In [ ]:
random.shuffle(pairs)
split = int(0.9 * len(pairs))
train_pairs, test_pairs = pairs[:split], pairs[split:]
print(f'Train: {len(train_pairs)}   Test: {len(test_pairs)}')


def indexes_from_sentence(lang, sentence):
    return [lang.word2index[w] for w in sentence.split(' ') if w in lang.word2index]


def tensor_from_sentence(lang, sentence):
    indexes = indexes_from_sentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1, -1)


def tensors_from_pair(pair):
    return (
        tensor_from_sentence(input_lang, pair[0]),
        tensor_from_sentence(output_lang, pair[1]),
    )

## 6 — Training loop (GPU-accelerated 🚀)

In [ ]:
import time
from torch import optim

encoder = EncoderRNN(input_lang.n_words, HIDDEN_SIZE).to(device)
decoder = AttnDecoderRNN(HIDDEN_SIZE, output_lang.n_words).to(device)

encoder_opt = optim.Adam(encoder.parameters(), lr=LEARNING_RATE)
decoder_opt = optim.Adam(decoder.parameters(), lr=LEARNING_RATE)
criterion = nn.NLLLoss()

print(f'Encoder params: {sum(p.numel() for p in encoder.parameters()):,}')
print(f'Decoder params: {sum(p.numel() for p in decoder.parameters()):,}')
print(f'Training on: {device}  |  {EPOCHS} epochs  |  {len(train_pairs)} pairs/epoch')
print('—' * 60)

losses = []
start = time.time()

for epoch in range(1, EPOCHS + 1):
    total_loss = 0
    random.shuffle(train_pairs)

    for i, pair in enumerate(train_pairs, start=1):
        input_tensor, target_tensor = tensors_from_pair(pair)
        encoder_opt.zero_grad()
        decoder_opt.zero_grad()

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, _ = decoder(
            encoder_outputs, encoder_hidden, target_tensor, device=device
        )

        loss = criterion(
            decoder_outputs.view(-1, decoder_outputs.size(-1)),
            target_tensor.view(-1),
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(encoder.parameters(), 5.0)
        torch.nn.utils.clip_grad_norm_(decoder.parameters(), 5.0)
        encoder_opt.step()
        decoder_opt.step()
        total_loss += loss.item()

        if i % PRINT_EVERY == 0 or i == len(train_pairs):
            print(f'  epoch {epoch} — {i}/{len(train_pairs)} — '
                  f'running loss {total_loss / i:.4f}')

    avg_loss = total_loss / len(train_pairs)
    losses.append(avg_loss)
    elapsed = time.time() - start
    print(f'✅ Epoch {epoch}/{EPOCHS} — loss {avg_loss:.4f} — '
          f'elapsed {elapsed:.0f}s')

print(f'\n🎉 Training complete in {time.time()-start:.0f}s')

## 7 — Training loss plot

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(losses)+1), losses, 'o-', color='#6366f1', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('NLL Loss')
plt.title('Training Loss over Epochs')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8 — Save models to Google Drive

Saves:
- `encoder.pt` — encoder state dict
- `decoder.pt` — decoder state dict
- `vocab.pkl` — vocabularies + hidden size
- **`translation_model.pkl`** — complete model dump (encoder + decoder + vocab, all in one)

In [ ]:
import pickle

os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)

# ── Individual .pt files (same format as your local project) ──────
torch.save(encoder.state_dict(), f'{DRIVE_MODEL_DIR}/encoder.pt')
torch.save(decoder.state_dict(), f'{DRIVE_MODEL_DIR}/decoder.pt')
print(f'✅ Saved encoder.pt & decoder.pt → {DRIVE_MODEL_DIR}/')

# ── Vocab pickle ──────────────────────────────────────────────────
with open(f'{DRIVE_MODEL_DIR}/vocab.pkl', 'wb') as f:
    pickle.dump({
        'input_lang': input_lang,
        'output_lang': output_lang,
        'hidden_size': HIDDEN_SIZE,
    }, f)
print(f'✅ Saved vocab.pkl → {DRIVE_MODEL_DIR}/')

# ── Full model .pkl dump (everything in one file) ────────────────
# Move models to CPU before pickling so the .pkl works anywhere
encoder_cpu = EncoderRNN(input_lang.n_words, HIDDEN_SIZE)
encoder_cpu.load_state_dict(encoder.state_dict())
encoder_cpu.eval()

decoder_cpu = AttnDecoderRNN(HIDDEN_SIZE, output_lang.n_words)
decoder_cpu.load_state_dict(decoder.state_dict())
decoder_cpu.eval()

model_bundle = {
    'encoder': encoder_cpu,
    'decoder': decoder_cpu,
    'input_lang': input_lang,
    'output_lang': output_lang,
    'hidden_size': HIDDEN_SIZE,
    'max_length': MAX_LENGTH,
    'epochs_trained': EPOCHS,
    'final_loss': losses[-1],
}

with open(f'{DRIVE_MODEL_DIR}/translation_model.pkl', 'wb') as f:
    pickle.dump(model_bundle, f)
print(f'✅ Saved translation_model.pkl → {DRIVE_MODEL_DIR}/')

# Also save training loss plot to Drive
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(losses)
ax.set_xlabel('Epoch')
ax.set_ylabel('NLL Loss')
ax.set_title('Training loss over epochs')
fig.savefig(f'{DRIVE_MODEL_DIR}/training_loss.png', dpi=150)
plt.close(fig)
print(f'✅ Saved training_loss.png → {DRIVE_MODEL_DIR}/')

print('\n📁 All saved files:')
for f in os.listdir(DRIVE_MODEL_DIR):
    size = os.path.getsize(f'{DRIVE_MODEL_DIR}/{f}')
    print(f'   {f:30s} {size/1024:.1f} KB')

## 9 — Quick evaluation (test translations)

In [ ]:
def evaluate(sentence):
    with torch.no_grad():
        input_tensor = tensor_from_sentence(
            input_lang, normalize_string(sentence)
        )
        encoder_outputs, encoder_hidden = encoder(input_tensor)
        input_len = input_tensor.size(1)
        max_decode = min(max(input_len + 3, 5), MAX_LENGTH)
        decoder_outputs, _ = decoder(
            encoder_outputs, encoder_hidden,
            max_len=max_decode, device=device,
        )
        _, topi = decoder_outputs.topk(1)
        words = []
        for idx in topi.squeeze():
            if idx.item() == EOS_token:
                break
            words.append(output_lang.index2word[idx.item()])
        return ' '.join(words)


print('\n🔤 Sample translations:')
test_sentences = [
    'I am learning french.',
    'Hello.',
    'How are you?',
    'Thank you.',
    'Good morning.',
    'I love you.',
    'Where is the bathroom?',
]

for sent in test_sentences:
    print(f'  EN: {sent}')
    print(f'  FR: {evaluate(sent)}\n')

print('\n📝 Random test-set samples:')
for pair in random.sample(test_pairs, min(5, len(test_pairs))):
    print(f'  > {pair[0]}')
    print(f'  = {pair[1]}')
    print(f'  < {evaluate(pair[0])}\n')

## 10 — How to use the saved `.pkl` model locally

```python
import pickle
import torch

with open('translation_model.pkl', 'rb') as f:
    bundle = pickle.load(f)

encoder = bundle['encoder']
decoder = bundle['decoder']
input_lang = bundle['input_lang']
output_lang = bundle['output_lang']

# Ready to translate!
```